In [1]:
import torch
import torch.nn as nn

# **Group Query Attention**

In [14]:
torch.manual_seed(123)
n_heads = 5
emb_size = 10
num_tokens = 4
head_dim = emb_size//n_heads
groups = 1

attn_w = torch.rand(n_heads ,num_tokens, num_tokens)
value = torch.rand(groups, num_tokens, head_dim)

attn_w[1] @ value



tensor([[[0.7970, 1.1594],
         [1.3151, 1.8083],
         [0.8249, 1.3113],
         [1.0347, 1.4482]]])

In [11]:
context_vectors = attn_w @ value
context_vectors

tensor([[[0.7171, 1.0958],
         [0.6234, 0.5669],
         [0.7747, 1.1512],
         [0.3228, 0.5943]],

        [[0.7970, 1.1594],
         [1.3151, 1.8083],
         [0.8249, 1.3113],
         [1.0347, 1.4482]],

        [[1.2387, 1.3736],
         [0.4073, 0.7353],
         [0.7290, 1.3483],
         [0.8539, 1.1331]],

        [[1.4456, 1.6226],
         [0.9719, 0.8017],
         [0.8320, 0.7701],
         [1.0082, 0.8714]],

        [[0.7047, 0.9180],
         [0.8173, 1.3682],
         [0.4240, 0.8076],
         [0.7954, 1.2972]]])

In [15]:
context_vectors.shape

torch.Size([5, 4, 2])

In [17]:
context_vectors = context_vectors.transpose(0,1)


In [18]:
context_vectors.shape

torch.Size([4, 5, 2])

In [21]:
context_vectors.contiguous().view(num_tokens, emb_size)

tensor([[0.7171, 1.0958, 0.7970, 1.1594, 1.2387, 1.3736, 1.4456, 1.6226, 0.7047,
         0.9180],
        [0.6234, 0.5669, 1.3151, 1.8083, 0.4073, 0.7353, 0.9719, 0.8017, 0.8173,
         1.3682],
        [0.7747, 1.1512, 0.8249, 1.3113, 0.7290, 1.3483, 0.8320, 0.7701, 0.4240,
         0.8076],
        [0.3228, 0.5943, 1.0347, 1.4482, 0.8539, 1.1331, 1.0082, 0.8714, 0.7954,
         1.2972]])

In [22]:
batch_size = 2
groups = 3
n_heads = 6
emb_size = 6
num_tokens = 4

group_size = n_heads//groups
head_dim = emb_size//n_heads
kv_out_dim = groups * head_dim

q_w = torch.rand(emb_size, emb_size)
k_w = torch.rand(emb_size, kv_out_dim)

In [39]:
temp_query = torch.rand((batch_size, num_tokens, emb_size))
temp_query

tensor([[[0.4596, 0.4947, 0.1836, 0.2010, 0.9603, 0.6861],
         [0.4209, 0.8046, 0.2621, 0.0638, 0.0036, 0.7032],
         [0.3051, 0.8070, 0.9271, 0.6647, 0.9296, 0.3848],
         [0.9357, 0.2616, 0.4344, 0.8323, 0.2410, 0.8815]],

        [[0.6226, 0.4902, 0.9279, 0.8751, 0.2943, 0.5485],
         [0.5583, 0.9096, 0.7810, 0.9049, 0.8048, 0.0649],
         [0.8322, 0.3672, 0.9012, 0.8146, 0.2077, 0.4474],
         [0.5746, 0.6429, 0.0369, 0.5224, 0.7605, 0.7823]]])

In [40]:
temp_query = temp_query.view(batch_size, num_tokens, n_heads, head_dim)
temp_query = temp_query.transpose(1,2)
temp_query = temp_query.contiguous().view(batch_size, group_size, groups, num_tokens, head_dim)
temp_query

tensor([[[[[0.4596],
           [0.4209],
           [0.3051],
           [0.9357]],

          [[0.4947],
           [0.8046],
           [0.8070],
           [0.2616]],

          [[0.1836],
           [0.2621],
           [0.9271],
           [0.4344]]],


         [[[0.2010],
           [0.0638],
           [0.6647],
           [0.8323]],

          [[0.9603],
           [0.0036],
           [0.9296],
           [0.2410]],

          [[0.6861],
           [0.7032],
           [0.3848],
           [0.8815]]]],



        [[[[0.6226],
           [0.5583],
           [0.8322],
           [0.5746]],

          [[0.4902],
           [0.9096],
           [0.3672],
           [0.6429]],

          [[0.9279],
           [0.7810],
           [0.9012],
           [0.0369]]],


         [[[0.8751],
           [0.9049],
           [0.8146],
           [0.5224]],

          [[0.2943],
           [0.8048],
           [0.2077],
           [0.7605]],

          [[0.5485],
           [0.0649],
    

In [23]:
query = torch.rand((batch_size, group_size, groups, num_tokens, head_dim))
query.shape

torch.Size([2, 2, 3, 4, 1])

In [27]:
key = torch.rand((batch_size, groups, num_tokens, head_dim))
key.unsqueeze_(1) #without new dim, the result of matmul would be incorrect
key.shape

torch.Size([2, 1, 3, 4, 1])

In [34]:
value = torch.rand((batch_size, groups, num_tokens, head_dim))
value.unsqueeze_(1)
value.shape

torch.Size([2, 1, 3, 4, 1])

In [31]:
attention_score = query @ key.transpose(-2,-1)
attention_score

tensor([[[[[0.1504, 0.6304, 0.5013, 0.5114],
           [0.2303, 0.9653, 0.7676, 0.7830],
           [0.1345, 0.5639, 0.4484, 0.4574],
           [0.0651, 0.2728, 0.2169, 0.2213]],

          [[0.1287, 0.0876, 0.1521, 0.0989],
           [0.3559, 0.2423, 0.4206, 0.2737],
           [0.0854, 0.0581, 0.1009, 0.0656],
           [0.0821, 0.0559, 0.0970, 0.0631]],

          [[0.0554, 0.1323, 0.1147, 0.0647],
           [0.1049, 0.2503, 0.2169, 0.1223],
           [0.1706, 0.4071, 0.3528, 0.1989],
           [0.2682, 0.6401, 0.5546, 0.3128]]],


         [[[0.1621, 0.6794, 0.5403, 0.5511],
           [0.2063, 0.8649, 0.6878, 0.7015],
           [0.0435, 0.1822, 0.1449, 0.1478],
           [0.1267, 0.5310, 0.4223, 0.4307]],

          [[0.0367, 0.0250, 0.0434, 0.0283],
           [0.5203, 0.3542, 0.6149, 0.4001],
           [0.3996, 0.2720, 0.4722, 0.3072],
           [0.6505, 0.4428, 0.7687, 0.5002]],

          [[0.0459, 0.1095, 0.0949, 0.0535],
           [0.2862, 0.6831, 0.5918, 0.3338]

In [32]:
attention_score.shape

torch.Size([2, 2, 3, 4, 4])

In [29]:
query[0,0,0] @ key[0,0,0].T

tensor([[0.1504, 0.6304, 0.5013, 0.5114],
        [0.2303, 0.9653, 0.7676, 0.7830],
        [0.1345, 0.5639, 0.4484, 0.4574],
        [0.0651, 0.2728, 0.2169, 0.2213]])

In [30]:
query[0,1,0] @ key[0,0,0].T

tensor([[0.1621, 0.6794, 0.5403, 0.5511],
        [0.2063, 0.8649, 0.6878, 0.7015],
        [0.0435, 0.1822, 0.1449, 0.1478],
        [0.1267, 0.5310, 0.4223, 0.4307]])

In [35]:
context_vectors_new = attention_score @ value
context_vectors_new.shape

torch.Size([2, 2, 3, 4, 1])

In [36]:
context_vectors_new

tensor([[[[[1.1105],
           [1.7004],
           [0.9933],
           [0.4806]],

          [[0.1551],
           [0.4291],
           [0.1029],
           [0.0990]],

          [[0.1279],
           [0.2420],
           [0.3936],
           [0.6188]]],


         [[[1.1968],
           [1.5235],
           [0.3210],
           [0.9354]],

          [[0.0443],
           [0.6273],
           [0.4817],
           [0.7842]],

          [[0.1058],
           [0.6603],
           [0.6758],
           [0.6627]]]],



        [[[[0.3850],
           [0.6433],
           [0.5620],
           [0.5883]],

          [[0.1735],
           [0.0431],
           [0.1832],
           [0.2203]],

          [[0.2567],
           [0.3432],
           [0.0332],
           [0.5228]]],


         [[[0.6304],
           [0.3946],
           [0.3164],
           [0.3505]],

          [[0.3012],
           [0.0217],
           [0.0424],
           [0.1238]],

          [[0.3417],
           [1.2202],
    

In [37]:
context_vectors_new = context_vectors_new.contiguous().view(batch_size, n_heads, num_tokens, head_dim)
context_vectors_new = context_vectors_new.transpose(1,2)
context_vectors_new = context_vectors_new.contiguous().view(batch_size, num_tokens, emb_size)
context_vectors_new.shape

torch.Size([2, 4, 6])

In [38]:
context_vectors_new

tensor([[[1.1105, 0.1551, 0.1279, 1.1968, 0.0443, 0.1058],
         [1.7004, 0.4291, 0.2420, 1.5235, 0.6273, 0.6603],
         [0.9933, 0.1029, 0.3936, 0.3210, 0.4817, 0.6758],
         [0.4806, 0.0990, 0.6188, 0.9354, 0.7842, 0.6627]],

        [[0.3850, 0.1735, 0.2567, 0.6304, 0.3012, 0.3417],
         [0.6433, 0.0431, 0.3432, 0.3946, 0.0217, 1.2202],
         [0.5620, 0.1832, 0.0332, 0.3164, 0.0424, 1.3738],
         [0.5883, 0.2203, 0.5228, 0.3505, 0.1238, 0.0240]]])

In [50]:
x = torch.triu(torch.ones(5, 5), diagonal=1)
x

tensor([[0., 1., 1., 1., 1.],
        [0., 0., 1., 1., 1.],
        [0., 0., 0., 1., 1.],
        [0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0.]])

In [88]:
###lets build GQA module
class GroupedQueryAttention(nn.Module):
  def __init__(self, in_dim, out_dim, num_heads, num_groups, context_length, dropout, qkv_bias=False):
    super().__init__()
    assert (out_dim % num_heads == 0), "out_dim is not divisible by num_heads"
    assert (num_heads % num_groups == 0), "num_heads is not divisible by groups"

    self.in_dim = in_dim
    self.out_dim = out_dim
    self.num_heads = num_heads
    self.head_dim = out_dim//num_heads
    self.num_groups = num_groups
    self.group_size = num_heads//num_groups
    self.kv_dim = self.head_dim * num_groups
    self.w_query = nn.Linear(in_dim, out_dim, bias=qkv_bias)
    self.w_key = nn.Linear(in_dim, self.kv_dim, bias=qkv_bias)
    self.w_value = nn.Linear(in_dim, self.kv_dim, bias=qkv_bias)
    self.out_proj = nn.Linear(out_dim, out_dim, bias=qkv_bias)
    self.drop = nn.Dropout(dropout)
    self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))

  def forward(self, x):
    batch_size, num_tokens, in_dim = x.shape
    query = self.w_query(x) #shape(batch_size, num_tokens, out_dim)
    key = self.w_key(x) #shape(batch_size, num_tokens, kv_dim)
    value = self.w_value(x) #shape(batch_size, num_tokens, kv_dim)

    query = query.view(batch_size, num_tokens, self.num_heads, self.head_dim)
    key = key.view(batch_size, num_tokens, self.num_groups, self.head_dim)
    value = value.view(batch_size, num_tokens, self.num_groups, self.head_dim)

    query = query.transpose(1,2)
    key = key.transpose(1,2)
    value = value.transpose(1,2)

    query = query.contiguous().view(batch_size, self.group_size, self.num_groups, num_tokens, self.head_dim)
    key.unsqueeze_(1)
    value.unsqueeze_(1)

    attention_weights = query @ key.transpose(-2,-1)
    attention_weights.masked_fill_(self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)
    attention_scores = torch.softmax((attention_weights/(key.shape[-1] ** 0.5)), dim=-1)

    attention_scores = self.drop(attention_scores)

    context_vectors = attention_scores @ value
    context_vectors = context_vectors.contiguous().view(batch_size, self.num_heads, num_tokens, self.head_dim)
    context_vectors = context_vectors.transpose(1,2)
    context_vectors = context_vectors.contiguous().view(batch_size, num_tokens, self.out_dim)

    attn_output = self.out_proj(context_vectors)

    return attn_output

In [95]:
torch.manual_seed(123)
batch_size = 2
num_tokens = 4
emb_size = 768
num_heads = 12
num_groups = 4
context_length = 1024

demo_inp = torch.rand((batch_size, num_tokens, emb_size))

gqa = GroupedQueryAttention(emb_size, emb_size, num_heads=num_heads, num_groups=num_groups, context_length=context_length, dropout=0)
demo_out = gqa(demo_inp)

In [96]:
demo_out

tensor([[[ 0.0106, -0.3444, -0.0366,  ..., -0.2449, -0.1972,  0.0950],
         [ 0.0389, -0.3119, -0.0444,  ..., -0.3244, -0.1529,  0.0868],
         [ 0.0343, -0.2934, -0.0935,  ..., -0.3400, -0.1362,  0.1756],
         [ 0.0684, -0.2907, -0.1291,  ..., -0.3061, -0.1115,  0.2063]],

        [[-0.0165, -0.1321, -0.1452,  ..., -0.0799, -0.0471,  0.1032],
         [ 0.0489, -0.2310, -0.1996,  ..., -0.2016, -0.0278,  0.1972],
         [ 0.0193, -0.2422, -0.1875,  ..., -0.2214,  0.0177,  0.1766],
         [ 0.0255, -0.2701, -0.1827,  ..., -0.2262, -0.0322,  0.1619]]],
       grad_fn=<UnsafeViewBackward0>)